In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

access_id = os.getenv("ODPS_ACCESS_ID")
access_key = os.getenv("ODPS_ACCESS_KEY")
project = os.getenv("ODPS_PROJECT")
endpoint = os.getenv("ODPS_ENDPOINT")

from odps import ODPS, options

o = ODPS(
    access_id=access_id,
    secret_access_key=access_key,
    project=project,
    endpoint=endpoint,
)

options.tunnel.use_instance_tunnel = True
options.tunnel.limit_instance_tunnel = False


In [ ]:
def q(sql):
    """Run SQL on DataWorks and return the result as a pandas DataFrame."""
    inst = o.execute_sql(sql)
    inst.wait_for_success()
    with inst.open_reader(tunnel=True) as reader:
        return reader.to_pandas()


print("Connected to project:", o.project)
print(os.getenv("ODPS_ACCESS_ID"))
print(os.getenv("ODPS_ACCESS_KEY"))

In [ ]:
lookback_dt = '2026-05-01'
train_start = '2026-07-01'
train_end = '2026-08-27'
oot_start = '2026-08-28'
oot_end = '2026-10-01'

In [ ]:
sql = f"""
WITH dispute_table AS (
  SELECT
    user_id,
    trans_id, 
    DATE_FORMAT(COALESCE(mexico_city_time, payment_created_at_cdmx), 'yyyy-MM-dd') AS transaction_time, 
    DATE_FORMAT(controversy_date_cdmx, 'yyyy-MM-dd') AS controversy_date_cdmx, 
    controversy_type, 
    reason_code, 
    payment_amount, 
    currency, 
    ROW_NUMBER() OVER (PARTITION BY trans_id, user_id ORDER BY dt ASC) AS rn
FROM basehub_mx.dwd_controversy_dispute_miclip_match_df
WHERE reason_code IN ('68', '41') AND trans_id IS NOT NULL
)

SELECT
    DATEDIFF(controversy_date_cdmx, transaction_time) AS days_claim,
    * ,
    CASE
      WHEN transaction_time >= '{train_start}' AND transaction_time <= '{train_end}' THEN 'training'
      WHEN transaction_time >= '{oot_start}'   AND transaction_time <= '{oot_end}'   THEN 'oot'
    END AS sample_type
FROM dispute_table
WHERE rn = 1
;
"""

instance = o.execute_sql(sql)
instance.wait_for_success()

In [ ]:
import pandas as pd

with instance.open_reader(tunnel=True) as reader:
    df = reader.to_pandas()

df.describe()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Pull data
data = df[['days_claim']]

# Calculate the quartiles (25th, 50th, 75th, and 90th percentiles)
q1, median, q3, p90 = np.percentile(data, [25, 50, 75, 90])

# Create the Seaborn histogram
plt.figure(figsize=(8, 4))
ax = sns.histplot(data, bins=30, kde=True, color="#257cbe", edgecolor="white", alpha=0.7)

# Add vertical lines for quartiles
ax.axvline(q1, color='#bb0e0e', linestyle='-', linewidth=2, label=f'Q1 (25th): {q1:.0f}')
ax.axvline(median, color='#d25f00', linestyle='--', linewidth=2, label=f'Median (50th): {median:.0f}')
ax.axvline(q3, color='#f0b603', linestyle='-', linewidth=2, label=f'Q3 (75th): {q3:.0f}')
ax.axvline(p90, color='#9dc2e5', linestyle='--', linewidth=2, label=f'p90 (90th): {p90:.0f}')

# 5. Remove the outer borders (spines)
sns.despine(top=True, right=True, left=False, bottom=False)

# 6. Customize labels & legend
plt.title('Lag in time for disputes distribution', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('days', fontsize=12)
plt.ylabel('volume', fontsize=12)
plt.legend(loc='upper right', frameon=False)
plt.grid(axis='y', alpha=0.2) # Light horizontal grid lines for readability

plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Pull data
data = df[['transaction_time']]
data['transaction_time'] = pd.to_datetime(data['transaction_time'], format='%Y-%m-%d')

# Sort data chronologically so the plot flows correctly
data = data.sort_values('transaction_time')

# Plot
plt.figure(figsize=(10, 5))
ax = sns.countplot(data=data, x='transaction_time', color='skyblue', edgecolor='white')

# Clean up borders and style
sns.despine(top=True, right=True, left=True, bottom=True)
plt.title('Observations per Date', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.grid(axis='y', alpha=0.2)

plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Pull data
data = df[['transaction_time']]
data['transaction_time'] = pd.to_datetime(data['transaction_time'], format='%Y-%m-%d')


# AGGREGATE: Group data by week and count observations
# 'W' counts weeks ending on Sunday. Use 'W-MON' if you prefer weeks starting on Monday.
weekly_counts = data.groupby(pd.Grouper(key='transaction_time', freq='W')).size().reset_index(name='count')

# FORMAT: Convert week timestamps to clean string labels (e.g., "Week of Oct 12")
weekly_counts['week_label'] = weekly_counts['transaction_time'].dt.strftime('%b %d')

# PLOT
plt.figure(figsize=(12, 5))
ax = sns.barplot(
    data=weekly_counts, 
    x='week_label', 
    y='count', 
    color='skyblue', 
    edgecolor='white'
)

# ADD LABELS TO THE COUNTS 
# padding=3 puts a small space between the number and the top of the bar
ax.bar_label(ax.containers[0], padding=3, fontsize=10, fontweight='bold', color='#333333')


# STYLE: Minimalist style with all borders removed
sns.despine(top=True, right=True, left=True, bottom=True)
plt.title('Observations per Week', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Week Starting/Ending', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.xticks(rotation=45, ha='right') # Rotates labels slightly so they don't overlap
plt.grid(axis='y', alpha=0.2)

plt.tight_layout()
plt.show()


In [ ]:
# 1. Ensure datetime formatting and grouping
data = df[['transaction_time']]
data['transaction_time'] = pd.to_datetime(data['transaction_time'], format='%Y-%m-%d')
weekly_counts = data.groupby(pd.Grouper(key='transaction_time', freq='W')).size().reset_index(name='Observations')

# Format the week dates into a clean string label
weekly_counts['Week Ending'] = weekly_counts['transaction_time'].dt.strftime('%Y-%m-%d')

# Keep only the columns you want to view and print
table_df = weekly_counts[['Week Ending', 'Observations']]
print(table_df.to_string(index=False))

In [ ]:
# Ensure datetime formatting and grouping
data = df[['transaction_time']]
data['transaction_time'] = pd.to_datetime(data['transaction_time'], format='%Y-%m-%d')
weekly_counts = data.groupby(pd.Grouper(key='transaction_time')).size().reset_index(name='observations')

# Keep only the columns you want to view and print
table_df = weekly_counts[['transaction_time', 'observations']]
table_df.to_csv('claims.csv')

In [ ]:
df.to_csv('transactions_claims.csv')